In [ ]:
"""
================================================================================
  CELL 1 — MASTER CONFIGURATION & DATA CENSUS
  Notebook: 01_WILLIE_DataForge.ipynb
================================================================================
  PURPOSE:
    • Define ALL project paths in one place (single source of truth)
    • Verify every expected directory exists
    • Count every image in every folder across all 3 datasets
    • Measure image dimensions (answers: what resolution are we working with?)
    • Report class distributions (answers: how imbalanced are we?)
  
  OUTPUT:
    • Config dict (CFG) used by ALL cells and ALL notebooks
    • Directory existence check (green/red)
    • Per-dataset, per-split, per-class image counts
    • Image dimension statistics (min, max, mean, mode)
    • Summary table
================================================================================
"""

import os
import sys
import json
import glob
import random
import warnings
from pathlib import Path
from datetime import datetime
from collections import defaultdict, Counter

import numpy as np
from PIL import Image
from tqdm.notebook import tqdm
import matplotlib.pyplot as plt
import matplotlib
matplotlib.rcParams['figure.dpi'] = 120
matplotlib.rcParams['font.size'] = 10
warnings.filterwarnings('ignore')

# ══════════════════════════════════════════════════════════════════════════════
# 1. MASTER CONFIGURATION — EVERY PATH LIVES HERE AND ONLY HERE
# ══════════════════════════════════════════════════════════════════════════════

PROJECT_ROOT = "."

CFG = {
    # ── Project roots ──
    "project_root":     PROJECT_ROOT,
    "data_root":        os.path.join(PROJECT_ROOT, "data"),
    "artifacts_root":   os.path.join(PROJECT_ROOT, "artifacts"),
    
    # ── FUSeg dataset ──
    "fuseg_root":       os.path.join(PROJECT_ROOT, "data", "FUSeg"),
    "fuseg_train_img":  os.path.join(PROJECT_ROOT, "data", "FUSeg", "train", "images"),
    "fuseg_train_lbl":  os.path.join(PROJECT_ROOT, "data", "FUSeg", "train", "labels"),
    "fuseg_val_img":    os.path.join(PROJECT_ROOT, "data", "FUSeg", "val", "images"),
    "fuseg_val_lbl":    os.path.join(PROJECT_ROOT, "data", "FUSeg", "val", "labels"),
    "fuseg_test_img":   os.path.join(PROJECT_ROOT, "data", "FUSeg", "test"),
    
    # ── AZH dataset ──
    "azh_root":         os.path.join(PROJECT_ROOT, "data", "AZH"),
    "azh_train_root":   os.path.join(PROJECT_ROOT, "data", "AZH", "train"),
    "azh_test_root":    os.path.join(PROJECT_ROOT, "data", "AZH", "test"),
    "azh_classes":      ["BG", "diabetic", "no wound", "pressure", "surgical", "venous"],
    
    # ── Medetec dataset ──
    "medetec_root":     os.path.join(PROJECT_ROOT, "data", "Medetec"),
    "medetec_classes":  ["diabetic", "pressure", "toes", "venous"],
    
    # ── Unified class taxonomy ──
    "wound_classes":    ["diabetic", "pressure", "surgical", "venous", "no_wound"],
    "num_classes":      5,
    "class_map": {
        # AZH mappings
        "diabetic":  0,
        "pressure":  1,
        "surgical":  2,
        "venous":    3,
        "BG":        4,  # → no_wound
        "no wound":  4,  # → no_wound
        # Medetec mappings
        "toes":      0,  # → diabetic (clinical: diabetic foot ulcer)
    },
    
    # ── Output directories (created as needed) ──
    "output_root":      os.path.join(PROJECT_ROOT, "artifacts", "willie_v2"),
    "manifests_dir":    os.path.join(PROJECT_ROOT, "artifacts", "willie_v2", "manifests"),
    "checkpoints_dir":  os.path.join(PROJECT_ROOT, "artifacts", "willie_v2", "checkpoints"),
    "logs_dir":         os.path.join(PROJECT_ROOT, "artifacts", "willie_v2", "logs"),
    "figures_dir":      os.path.join(PROJECT_ROOT, "artifacts", "willie_v2", "figures"),
    "cache_dir":        os.path.join(PROJECT_ROOT, "artifacts", "willie_v2", "cache"),
    
    # ── Detection config ──
    "det_imgsz":        512,
    "det_model":        "rtdetr-l",       # primary detector
    "det_model_cmp":    "yolov8m.pt",     # comparison detector
    "det_epochs":       80,
    "det_batch":        16,
    "det_conf_thr":     0.25,
    
    # ── SAM2 config ──
    "sam2_variant":     "sam2_hiera_small",
    "sam2_topk":        3,
    "sam2_score_thr":   0.8,
    
    # ── Transformer config (shared defaults, per-variant overrides later) ──
    "seed":             42,
    "num_workers":      4,
    "pin_memory":       True,
    
    # ── Image extensions to scan ──
    "img_exts":         {".jpg", ".jpeg", ".png", ".bmp", ".tif", ".tiff", ".webp"},
}

# ── Create output directories ──
for dir_key in ["output_root", "manifests_dir", "checkpoints_dir", "logs_dir", "figures_dir", "cache_dir"]:
    os.makedirs(CFG[dir_key], exist_ok=True)

# ── Save config to JSON for other notebooks to load ──
cfg_save_path = os.path.join(CFG["output_root"], "config.json")
with open(cfg_save_path, 'w') as f:
    # Convert sets to lists for JSON serialization
    cfg_serializable = {k: list(v) if isinstance(v, set) else v for k, v in CFG.items()}
    json.dump(cfg_serializable, f, indent=2)

print("=" * 80)
print("  Willie v2 — Master Configuration Loaded")
print(f"  Timestamp: {datetime.now().strftime('%Y-%m-%d %H:%M:%S')}")
print(f"  Config saved to: {cfg_save_path}")
print("=" * 80)

# ══════════════════════════════════════════════════════════════════════════════
# 2. DIRECTORY EXISTENCE VERIFICATION
# ══════════════════════════════════════════════════════════════════════════════

print("\n📂 DIRECTORY VERIFICATION")
print("-" * 80)

check_dirs = {
    "Project Root":         CFG["project_root"],
    "Data Root":            CFG["data_root"],
    # FUSeg
    "FUSeg Root":           CFG["fuseg_root"],
    "FUSeg Train Images":   CFG["fuseg_train_img"],
    "FUSeg Train Labels":   CFG["fuseg_train_lbl"],
    "FUSeg Val Images":     CFG["fuseg_val_img"],
    "FUSeg Val Labels":     CFG["fuseg_val_lbl"],
    "FUSeg Test Images":    CFG["fuseg_test_img"],
    # AZH
    "AZH Root":             CFG["azh_root"],
    "AZH Train Root":       CFG["azh_train_root"],
    "AZH Test Root":        CFG["azh_test_root"],
    # Medetec
    "Medetec Root":         CFG["medetec_root"],
}

# Add AZH per-class dirs
for split in ["Train", "Test"]:
    for cls in CFG["azh_classes"]:
        key = f"AZH {split}/{cls}"
        root_key = "azh_train_root" if split == "Train" else "azh_test_root"
        check_dirs[key] = os.path.join(CFG[root_key], cls)

# Add Medetec per-class dirs
for cls in CFG["medetec_classes"]:
    check_dirs[f"Medetec/{cls}"] = os.path.join(CFG["medetec_root"], cls)

all_exist = True
for name, path in check_dirs.items():
    exists = os.path.isdir(path)
    status = "✅" if exists else "❌ MISSING"
    if not exists:
        all_exist = False
    print(f"  {status}  {name:<30s}  →  {path}")

if all_exist:
    print(f"\n  ✅ ALL {len(check_dirs)} DIRECTORIES VERIFIED")
else:
    print(f"\n  ⚠️  SOME DIRECTORIES MISSING — check paths above!")

# ══════════════════════════════════════════════════════════════════════════════
# 3. COUNT EVERY IMAGE IN EVERY FOLDER
# ══════════════════════════════════════════════════════════════════════════════

def count_images(folder_path):
    """Count image files in a directory (non-recursive)."""
    if not os.path.isdir(folder_path):
        return 0, []
    files = []
    for f in os.listdir(folder_path):
        if os.path.splitext(f)[1].lower() in CFG["img_exts"]:
            files.append(os.path.join(folder_path, f))
    return len(files), files

def count_masks(folder_path):
    """Count mask/label files in a directory."""
    if not os.path.isdir(folder_path):
        return 0
    count = 0
    for f in os.listdir(folder_path):
        ext = os.path.splitext(f)[1].lower()
        if ext in CFG["img_exts"] or ext == ".txt":
            count += 1
    return count

# ── FUSeg Census ──
print("\n\n📊 DATASET CENSUS: FUSeg")
print("-" * 80)

fuseg_census = {}
for split, img_dir, lbl_dir in [
    ("train", CFG["fuseg_train_img"], CFG["fuseg_train_lbl"]),
    ("val",   CFG["fuseg_val_img"],   CFG["fuseg_val_lbl"]),
    ("test",  CFG["fuseg_test_img"],  None),
]:
    n_img, img_files = count_images(img_dir)
    n_lbl = count_masks(lbl_dir) if lbl_dir else "N/A"
    fuseg_census[split] = {"n_images": n_img, "n_labels": n_lbl, "files": img_files}
    lbl_str = str(n_lbl)
    match_str = ""
    if isinstance(n_lbl, int) and n_img > 0:
        match_str = " ✅ matched" if n_img == n_lbl else f" ⚠️ MISMATCH (diff={abs(n_img - n_lbl)})"
    print(f"  {split:<8s}:  {n_img:>5d} images  |  {lbl_str:>5s} labels{match_str}")

print(f"  {'TOTAL':<8s}:  {sum(v['n_images'] for v in fuseg_census.values()):>5d} images")

# ── AZH Census ──
print("\n\n📊 DATASET CENSUS: AZH")
print("-" * 80)

azh_census = {"Train": {}, "Test": {}}
for split in ["Train", "Test"]:
    root = CFG["azh_train_root"] if split == "Train" else CFG["azh_test_root"]
    total = 0
    for cls in CFG["azh_classes"]:
        cls_dir = os.path.join(root, cls)
        n, files = count_images(cls_dir)
        azh_census[split][cls] = {"count": n, "files": files}
        total += n
        # Map to unified class
        unified = CFG["wound_classes"][CFG["class_map"][cls]]
        print(f"  {split:<6s}/ {cls:<12s}:  {n:>5d} images  →  unified class: {unified}")
    azh_census[split]["_total"] = total
    print(f"  {split:<6s}/ {'TOTAL':<12s}:  {total:>5d} images")
    print()

# ── Medetec Census ──
print("\n📊 DATASET CENSUS: Medetec")
print("-" * 80)

medetec_census = {}
total_medetec = 0
for cls in CFG["medetec_classes"]:
    cls_dir = os.path.join(CFG["medetec_root"], cls)
    n, files = count_images(cls_dir)
    medetec_census[cls] = {"count": n, "files": files}
    total_medetec += n
    unified = CFG["wound_classes"][CFG["class_map"][cls]]
    print(f"  {cls:<12s}:  {n:>5d} images  →  unified class: {unified}")

print(f"  {'TOTAL':<12s}:  {total_medetec:>5d} images")

# ══════════════════════════════════════════════════════════════════════════════
# 4. IMAGE DIMENSION ANALYSIS (Answers Q3: What resolution are we working with?)
# ══════════════════════════════════════════════════════════════════════════════

print("\n\n📐 IMAGE DIMENSION ANALYSIS")
print("-" * 80)

def sample_dimensions(file_list, max_samples=100, dataset_name=""):
    """Sample image dimensions from a list of files."""
    if not file_list:
        return {"widths": [], "heights": [], "n_sampled": 0}
    
    sample = file_list if len(file_list) <= max_samples else random.sample(file_list, max_samples)
    widths, heights = [], []
    
    for fpath in sample:
        try:
            with Image.open(fpath) as img:
                w, h = img.size
                widths.append(w)
                heights.append(h)
        except Exception:
            pass
    
    return {"widths": widths, "heights": heights, "n_sampled": len(widths)}

# Collect all files per dataset
all_fuseg_files = []
for split_data in fuseg_census.values():
    all_fuseg_files.extend(split_data["files"])

all_azh_files = []
for split in ["Train", "Test"]:
    for cls in CFG["azh_classes"]:
        all_azh_files.extend(azh_census[split][cls]["files"])

all_medetec_files = []
for cls in CFG["medetec_classes"]:
    all_medetec_files.extend(medetec_census[cls]["files"])

dim_results = {}
for dname, flist in [("FUSeg", all_fuseg_files), ("AZH", all_azh_files), ("Medetec", all_medetec_files)]:
    dims = sample_dimensions(flist, max_samples=200, dataset_name=dname)
    dim_results[dname] = dims
    
    if dims["widths"]:
        w_arr = np.array(dims["widths"])
        h_arr = np.array(dims["heights"])
        w_mode = Counter(dims["widths"]).most_common(1)[0]
        h_mode = Counter(dims["heights"]).most_common(1)[0]
        
        print(f"\n  {dname} (sampled {dims['n_sampled']} images):")
        print(f"    Width  — min: {w_arr.min()}, max: {w_arr.max()}, "
              f"mean: {w_arr.mean():.0f}, mode: {w_mode[0]} (×{w_mode[1]})")
        print(f"    Height — min: {h_arr.min()}, max: {h_arr.max()}, "
              f"mean: {h_arr.mean():.0f}, mode: {h_mode[0]} (×{h_mode[1]})")
        
        # Unique dimension combos
        combos = Counter(zip(dims["widths"], dims["heights"]))
        print(f"    Unique (W×H) combos: {len(combos)}")
        for (w, h), cnt in combos.most_common(5):
            print(f"      {w}×{h}: {cnt} images")
    else:
        print(f"\n  {dname}: No images found or could not read.")

# ══════════════════════════════════════════════════════════════════════════════
# 5. UNIFIED CLASS DISTRIBUTION — What the transformer will see
# ══════════════════════════════════════════════════════════════════════════════

print("\n\n📊 UNIFIED CLASS DISTRIBUTION (All Datasets Combined)")
print("-" * 80)

unified_counts = defaultdict(int)
unified_sources = defaultdict(list)

# AZH contributions (Train + Test)
for split in ["Train", "Test"]:
    for cls in CFG["azh_classes"]:
        unified_cls = CFG["wound_classes"][CFG["class_map"][cls]]
        n = azh_census[split][cls]["count"]
        unified_counts[unified_cls] += n
        if n > 0:
            unified_sources[unified_cls].append(f"AZH-{split}/{cls}({n})")

# Medetec contributions
for cls in CFG["medetec_classes"]:
    unified_cls = CFG["wound_classes"][CFG["class_map"][cls]]
    n = medetec_census[cls]["count"]
    unified_counts[unified_cls] += n
    if n > 0:
        unified_sources[unified_cls].append(f"Medetec/{cls}({n})")

# FUSeg: wound images → we don't have wound TYPE labels, only masks
# FUSeg empty masks → no_wound
fuseg_wound_count = 0
fuseg_nowound_count = 0
for split in ["train", "val"]:
    for fpath in fuseg_census[split]["files"]:
        # We'll count empty masks in the next cell (need to read masks)
        # For now, mark all FUSeg as "wound (type unknown)" — they feed detection+segmentation, not classification
        fuseg_wound_count += 1

print("  Classification classes (for transformer training):")
print(f"  {'Class':<15s} {'Count':>8s}   Sources")
print(f"  {'─'*15:<15s} {'─'*8:>8s}   {'─'*45}")

total_cls = 0
for cls_name in CFG["wound_classes"]:
    n = unified_counts[cls_name]
    sources = ", ".join(unified_sources[cls_name]) if unified_sources[cls_name] else "—"
    print(f"  {cls_name:<15s} {n:>8d}   {sources}")
    total_cls += n

print(f"  {'─'*15:<15s} {'─'*8:>8s}")
print(f"  {'TOTAL':<15s} {total_cls:>8d}")

print(f"\n  Note: FUSeg has {fuseg_wound_count} wound images with masks but NO wound-type labels.")
print(f"        FUSeg → used for DETECTION + SEGMENTATION training (Stage 1 & 2).")
print(f"        AZH + Medetec → used for CLASSIFICATION training (Stage 3).")
print(f"        AZH negatives (BG + no wound) → also used as detection negatives.")

# ══════════════════════════════════════════════════════════════════════════════
# 6. FUSeg MASK AUDIT — Count empty masks (true negatives)
# ══════════════════════════════════════════════════════════════════════════════

print("\n\n🔍 FUSeg MASK AUDIT — Counting Empty Masks (True Negatives)")
print("-" * 80)

empty_mask_counts = {}
for split in ["train", "val"]:
    lbl_dir = CFG[f"fuseg_{split}_lbl"]
    if not os.path.isdir(lbl_dir):
        print(f"  {split}: label directory not found, skipping.")
        continue
    
    label_files = sorted([f for f in os.listdir(lbl_dir) 
                          if os.path.splitext(f)[1].lower() in CFG["img_exts"]])
    
    n_empty = 0
    n_total = len(label_files)
    
    for lf in tqdm(label_files, desc=f"  Auditing {split} masks", leave=True):
        try:
            mask = np.array(Image.open(os.path.join(lbl_dir, lf)))
            # Convert to single channel if needed (masks may be 3-channel PNG)
            if mask.ndim == 3:
                mask = mask[:, :, 0]
            if mask.max() == 0:
                n_empty += 1
        except Exception as e:
            print(f"    ⚠️ Could not read {lf}: {e}")
    
    empty_mask_counts[split] = {"empty": n_empty, "total": n_total, "wound": n_total - n_empty}
    print(f"  {split:<8s}: {n_total} total masks | {n_empty} empty (no wound) | {n_total - n_empty} with wound")

# ══════════════════════════════════════════════════════════════════════════════
# 7. SUMMARY TABLE
# ══════════════════════════════════════════════════════════════════════════════

print("\n\n" + "=" * 80)
print("  📋 COMPLETE DATA SUMMARY")
print("=" * 80)

print(f"""
  ┌─────────────────────────────────────────────────────────────────────────┐
  │                        DATASET OVERVIEW                                │
  ├──────────────┬──────────┬──────────────────────────────────────────────┤
  │ Dataset      │ Images   │ Role in Pipeline                            │
  ├──────────────┼──────────┼──────────────────────────────────────────────┤
  │ FUSeg        │ {sum(v['n_images'] for v in fuseg_census.values()):>6d}  │ Detection (Stage 1) + Segmentation (Stage 2) │
  │ AZH          │ {sum(azh_census[s]['_total'] for s in ['Train','Test']):>6d}  │ Classification (Stage 3) + Det. negatives    │
  │ Medetec      │ {total_medetec:>6d}  │ Classification (Stage 3) — extra training    │
  └──────────────┴──────────┴──────────────────────────────────────────────┘

  Config saved: {cfg_save_path}
  Ready for Cell 2: Data Visualization + Augmentation Preview
""")

# Save census results for later cells
census_save = {
    "fuseg": {k: {"n_images": v["n_images"], "n_labels": v["n_labels"]} for k, v in fuseg_census.items()},
    "azh": {split: {cls: azh_census[split][cls]["count"] for cls in CFG["azh_classes"]} for split in ["Train", "Test"]},
    "medetec": {cls: medetec_census[cls]["count"] for cls in CFG["medetec_classes"]},
    "unified_class_counts": dict(unified_counts),
    "fuseg_empty_masks": empty_mask_counts,
    "image_dimensions": {
        dname: {
            "widths_summary": {"min": int(np.min(d["widths"])), "max": int(np.max(d["widths"])), 
                               "mean": float(np.mean(d["widths"]))} if d["widths"] else {},
            "heights_summary": {"min": int(np.min(d["heights"])), "max": int(np.max(d["heights"])),
                                "mean": float(np.mean(d["heights"]))} if d["heights"] else {},
        }
        for dname, d in dim_results.items()
    }
}

census_path = os.path.join(CFG["manifests_dir"], "data_census.json")
with open(census_path, 'w') as f:
    json.dump(census_save, f, indent=2)
print(f"  Census data saved: {census_path}")

  WILLIE v2 — Master Configuration Loaded
  Timestamp: 2026-02-12 11:34:04
  Config saved to: artifacts/willie_v2/config.json

📂 DIRECTORY VERIFICATION
--------------------------------------------------------------------------------
  ✅  Project Root                    →  .
  ✅  Data Root                       →  data
  ✅  FUSeg Root                      →  data/FUSeg
  ✅  FUSeg Train Images              →  data/FUSeg/train/images
  ✅  FUSeg Train Labels              →  data/FUSeg/train/labels
  ✅  FUSeg Val Images                →  data/FUSeg/val/images
  ✅  FUSeg Val Labels                →  data/FUSeg/val/labels
  ✅  FUSeg Test Images               →  data/FUSeg/test
  ✅  AZH Root                        →  data/AZH
  ✅  AZH Train Root                  →  data/AZH/train
  ✅  AZH Test Root                   →  data/AZH/test
  ✅  Medetec Root                    →  data/Medetec
  ✅  AZH Train/BG                    →  data/AZH/train/BG
  ✅  AZH Train/diabetic              →  data/AZH/tra

  Auditing train masks:   0%|          | 0/610 [00:00<?, ?it/s]

  train   : 610 total masks | 10 empty (no wound) | 600 with wound


  Auditing val masks:   0%|          | 0/400 [00:00<?, ?it/s]

  val     : 400 total masks | 14 empty (no wound) | 386 with wound


  📋 COMPLETE DATA SUMMARY

  ┌─────────────────────────────────────────────────────────────────────────┐
  │                        DATASET OVERVIEW                                │
  ├──────────────┬──────────┬──────────────────────────────────────────────┤
  │ Dataset      │ Images   │ Role in Pipeline                            │
  ├──────────────┼──────────┼──────────────────────────────────────────────┤
  │ FUSeg        │   1010  │ Detection (Stage 1) + Segmentation (Stage 2) │
  │ AZH          │    930  │ Classification (Stage 3) + Det. negatives    │
  │ Medetec      │    384  │ Classification (Stage 3) — extra training    │
  └──────────────┴──────────┴──────────────────────────────────────────────┘

  Config saved: artifacts/willie_v2/config.json
  Ready for Cell 2: Data Visualization + Augmentation Preview

  Census data saved: artifacts/willie_v2/manifests/data_census.json


In [ ]:
"""
================================================================================
  CELL 2 — INVESTIGATE FUSeg TEST + FULL DATA VISUALIZATION
  Notebook: 01_WILLIE_DataForge.ipynb
================================================================================
  PURPOSE:
    • Investigate FUSeg test folder (0 images found — expected 200)
    • Visualize sample images from every dataset and class
    • Plot class distributions (bar charts)
    • Plot image dimension distributions per dataset
    • Visualize FUSeg masks with overlays on wound images
    • Save all figures to artifacts/willie_v2/figures/

  DEPENDS ON: Cell 1 (CFG dict must exist in memory)
================================================================================
"""

import os
import json
import random
import numpy as np
from pathlib import Path
from collections import Counter, defaultdict

from PIL import Image
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
import matplotlib.patches as mpatches
from tqdm.notebook import tqdm

# Verify CFG is loaded from Cell 1
assert 'CFG' in dir(), "❌ Run Cell 1 first — CFG not found in memory."
print("✅ CFG loaded from Cell 1\n")

# ══════════════════════════════════════════════════════════════════════════════
# 1. INVESTIGATE FUSeg TEST FOLDER — WHERE ARE THE 200 IMAGES?
# ══════════════════════════════════════════════════════════════════════════════

print("🔍 INVESTIGATING FUSeg TEST FOLDER")
print("-" * 80)

fuseg_test_dir = CFG["fuseg_test_img"]  # /project/.../data/FUSeg/test

if os.path.isdir(fuseg_test_dir):
    # List everything in the test directory
    all_entries = sorted(os.listdir(fuseg_test_dir))
    
    # Separate files and directories
    files_in_test = [e for e in all_entries if os.path.isfile(os.path.join(fuseg_test_dir, e))]
    dirs_in_test = [e for e in all_entries if os.path.isdir(os.path.join(fuseg_test_dir, e))]
    
    # Count image files directly in test/
    img_files_direct = [f for f in files_in_test if os.path.splitext(f)[1].lower() in CFG["img_exts"]]
    non_img_files = [f for f in files_in_test if os.path.splitext(f)[1].lower() not in CFG["img_exts"]]
    
    print(f"  Path: {fuseg_test_dir}")
    print(f"  Total entries: {len(all_entries)}")
    print(f"  ├── Subdirectories: {len(dirs_in_test)}")
    if dirs_in_test:
        for d in dirs_in_test[:10]:
            subdir_path = os.path.join(fuseg_test_dir, d)
            sub_files = os.listdir(subdir_path)
            sub_imgs = [f for f in sub_files if os.path.splitext(f)[1].lower() in CFG["img_exts"]]
            print(f"  │   └── {d}/  ({len(sub_imgs)} images, {len(sub_files)} total files)")
        if len(dirs_in_test) > 10:
            print(f"  │   └── ... and {len(dirs_in_test) - 10} more subdirectories")
    
    print(f"  ├── Image files (direct): {len(img_files_direct)}")
    if img_files_direct:
        exts = Counter(os.path.splitext(f)[1].lower() for f in img_files_direct)
        print(f"  │   Extensions: {dict(exts)}")
        print(f"  │   First 5: {img_files_direct[:5]}")
    
    print(f"  └── Non-image files: {len(non_img_files)}")
    if non_img_files:
        non_img_exts = Counter(os.path.splitext(f)[1].lower() for f in non_img_files)
        print(f"      Extensions: {dict(non_img_exts)}")
        print(f"      First 5: {non_img_files[:5]}")
    
    # Recursively search for ALL images under test/
    print(f"\n  Recursive search under {fuseg_test_dir}:")
    all_test_images = []
    for root, subdirs, fnames in os.walk(fuseg_test_dir):
        for fname in fnames:
            if os.path.splitext(fname)[1].lower() in CFG["img_exts"]:
                all_test_images.append(os.path.join(root, fname))
    
    print(f"  Total images found (recursive): {len(all_test_images)}")
    
    if all_test_images:
        # Show where they are
        test_subdirs = defaultdict(int)
        for p in all_test_images:
            rel = os.path.relpath(os.path.dirname(p), fuseg_test_dir)
            test_subdirs[rel] += 1
        for subdir, count in sorted(test_subdirs.items()):
            print(f"    {subdir}: {count} images")
        
        # Update config with correct path
        if len(img_files_direct) == 0 and len(dirs_in_test) > 0:
            # Images are in subdirectories — find the one with images
            best_subdir = max(test_subdirs.items(), key=lambda x: x[1])
            if best_subdir[0] == ".":
                corrected_path = fuseg_test_dir
            else:
                corrected_path = os.path.join(fuseg_test_dir, best_subdir[0])
            print(f"\n  ✅ RESOLVED: FUSeg test images found at: {corrected_path}")
            print(f"     Updating CFG['fuseg_test_img'] for subsequent cells.")
            CFG["fuseg_test_img"] = corrected_path
        elif len(img_files_direct) > 0:
            print(f"\n  ✅ Images are directly in test/ — original path is correct.")
        
        # Check dimensions of test images
        sample_test = random.sample(all_test_images, min(20, len(all_test_images)))
        test_dims = []
        for fpath in sample_test:
            try:
                with Image.open(fpath) as img:
                    test_dims.append(img.size)
            except:
                pass
        if test_dims:
            w_arr = [d[0] for d in test_dims]
            h_arr = [d[1] for d in test_dims]
            print(f"     Dimensions (sampled {len(test_dims)}): {Counter(test_dims).most_common(3)}")
    else:
        print("\n  ⚠️  NO IMAGES FOUND anywhere under FUSeg/test/")
        print("     Possible reasons:")
        print("     1. Test images haven't been downloaded/extracted yet")
        print("     2. They're stored in a different location")
        print("     3. Different file extensions than expected")
        print(f"     Expected extensions: {CFG['img_exts']}")
        
        # Check if there are ANY files at all
        all_test_files = []
        for root, subdirs, fnames in os.walk(fuseg_test_dir):
            all_test_files.extend(fnames)
        if all_test_files:
            ext_counts = Counter(os.path.splitext(f)[1].lower() for f in all_test_files)
            print(f"\n     Files found with extensions: {dict(ext_counts)}")
else:
    print(f"  ❌ Directory does not exist: {fuseg_test_dir}")

# ══════════════════════════════════════════════════════════════════════════════
# 2. HELPER — LOAD AND DISPLAY IMAGES
# ══════════════════════════════════════════════════════════════════════════════

def load_image_safe(fpath, max_size=512):
    """Load image, convert to RGB, optionally resize for display."""
    try:
        img = Image.open(fpath).convert("RGB")
        # Resize for display if too large
        if max(img.size) > max_size:
            img.thumbnail((max_size, max_size), Image.Resampling.LANCZOS)
        return np.array(img)
    except Exception as e:
        print(f"  ⚠️ Could not load {fpath}: {e}")
        return None

def get_sample_files(folder, n=3):
    """Get n random image files from a folder."""
    if not os.path.isdir(folder):
        return []
    imgs = [os.path.join(folder, f) for f in os.listdir(folder) 
            if os.path.splitext(f)[1].lower() in CFG["img_exts"]]
    if not imgs:
        return []
    return random.sample(imgs, min(n, len(imgs)))

random.seed(CFG["seed"])

# ══════════════════════════════════════════════════════════════════════════════
# 3. VISUALIZATION PANEL 1: SAMPLE IMAGES FROM EVERY DATASET + CLASS
# ══════════════════════════════════════════════════════════════════════════════

print("\n\n🖼️  PANEL 1: SAMPLE IMAGES FROM ALL DATASETS")
print("-" * 80)

# --- FUSeg Samples (image + mask overlay) ---
fig, axes = plt.subplots(2, 5, figsize=(20, 8))
fig.suptitle("FUSeg — Wound Images with Mask Overlays", fontsize=16, fontweight='bold', y=1.02)

fuseg_train_samples = get_sample_files(CFG["fuseg_train_img"], n=5)
for idx, fpath in enumerate(fuseg_train_samples):
    fname = os.path.basename(fpath)
    img = load_image_safe(fpath)
    if img is None:
        continue
    
    # Load corresponding mask
    mask_path = os.path.join(CFG["fuseg_train_lbl"], fname)
    # Try common mask naming patterns
    if not os.path.exists(mask_path):
        # Try without extension matching
        base = os.path.splitext(fname)[0]
        for ext in [".png", ".jpg", ".bmp", ".tif"]:
            candidate = os.path.join(CFG["fuseg_train_lbl"], base + ext)
            if os.path.exists(candidate):
                mask_path = candidate
                break
    
    # Row 1: Original image
    axes[0, idx].imshow(img)
    axes[0, idx].set_title(f"{fname[:20]}...", fontsize=8)
    axes[0, idx].axis("off")
    
    # Row 2: Image with mask overlay
    if os.path.exists(mask_path):
        mask = np.array(Image.open(mask_path))
        if mask.ndim == 3:
            mask = mask[:, :, 0]
        # Resize mask to match image if needed
        if mask.shape[:2] != img.shape[:2]:
            mask = np.array(Image.fromarray(mask).resize((img.shape[1], img.shape[0]), Image.NEAREST))
        
        # Create overlay
        overlay = img.copy()
        wound_mask = mask > 0
        overlay[wound_mask] = (overlay[wound_mask] * 0.5 + np.array([255, 0, 0]) * 0.5).astype(np.uint8)
        axes[1, idx].imshow(overlay)
        
        wound_pct = wound_mask.sum() / wound_mask.size * 100
        axes[1, idx].set_title(f"Mask overlay ({wound_pct:.1f}% wound)", fontsize=8)
    else:
        axes[1, idx].imshow(img)
        axes[1, idx].set_title("Mask not found", fontsize=8, color='red')
    axes[1, idx].axis("off")

plt.tight_layout()
plt.savefig(os.path.join(CFG["figures_dir"], "panel1_fuseg_samples.png"), dpi=150, bbox_inches='tight')
plt.show()
print(f"  Saved: {os.path.join(CFG['figures_dir'], 'panel1_fuseg_samples.png')}")

# --- AZH Samples (one row per class, 3 samples each) ---
fig, axes = plt.subplots(len(CFG["azh_classes"]), 4, figsize=(16, len(CFG["azh_classes"]) * 3))
fig.suptitle("AZH Dataset — Samples per Class", fontsize=16, fontweight='bold', y=1.01)

for row, cls in enumerate(CFG["azh_classes"]):
    cls_dir = os.path.join(CFG["azh_train_root"], cls)
    samples = get_sample_files(cls_dir, n=3)
    unified = CFG["wound_classes"][CFG["class_map"][cls]]
    
    # Label column
    axes[row, 0].text(0.5, 0.5, f"{cls}\n→ {unified}", transform=axes[row, 0].transAxes,
                      ha='center', va='center', fontsize=12, fontweight='bold',
                      bbox=dict(boxstyle='round,pad=0.5', facecolor='lightblue', alpha=0.8))
    axes[row, 0].axis("off")
    
    for col, fpath in enumerate(samples):
        img = load_image_safe(fpath)
        if img is not None:
            axes[row, col + 1].imshow(img)
            h, w = img.shape[:2]
            axes[row, col + 1].set_title(f"{w}×{h}", fontsize=8, color='gray')
        axes[row, col + 1].axis("off")
    
    # Fill empty slots
    for col in range(len(samples) + 1, 4):
        axes[row, col].axis("off")

plt.tight_layout()
plt.savefig(os.path.join(CFG["figures_dir"], "panel1_azh_samples.png"), dpi=150, bbox_inches='tight')
plt.show()
print(f"  Saved: {os.path.join(CFG['figures_dir'], 'panel1_azh_samples.png')}")

# --- Medetec Samples ---
fig, axes = plt.subplots(len(CFG["medetec_classes"]), 4, figsize=(16, len(CFG["medetec_classes"]) * 3))
fig.suptitle("Medetec Dataset — Samples per Class", fontsize=16, fontweight='bold', y=1.02)

for row, cls in enumerate(CFG["medetec_classes"]):
    cls_dir = os.path.join(CFG["medetec_root"], cls)
    samples = get_sample_files(cls_dir, n=3)
    unified = CFG["wound_classes"][CFG["class_map"][cls]]
    
    axes[row, 0].text(0.5, 0.5, f"{cls}\n→ {unified}", transform=axes[row, 0].transAxes,
                      ha='center', va='center', fontsize=12, fontweight='bold',
                      bbox=dict(boxstyle='round,pad=0.5', facecolor='lightyellow', alpha=0.8))
    axes[row, 0].axis("off")
    
    for col, fpath in enumerate(samples):
        img = load_image_safe(fpath)
        if img is not None:
            axes[row, col + 1].imshow(img)
            h, w = img.shape[:2]
            axes[row, col + 1].set_title(f"{w}×{h}", fontsize=8, color='gray')
        axes[row, col + 1].axis("off")
    
    for col in range(len(samples) + 1, 4):
        axes[row, col].axis("off")

plt.tight_layout()
plt.savefig(os.path.join(CFG["figures_dir"], "panel1_medetec_samples.png"), dpi=150, bbox_inches='tight')
plt.show()
print(f"  Saved: {os.path.join(CFG['figures_dir'], 'panel1_medetec_samples.png')}")

# ══════════════════════════════════════════════════════════════════════════════
# 4. VISUALIZATION PANEL 2: CLASS DISTRIBUTION BAR CHARTS
# ══════════════════════════════════════════════════════════════════════════════

print("\n\n📊 PANEL 2: CLASS DISTRIBUTIONS")
print("-" * 80)

fig, axes = plt.subplots(1, 3, figsize=(20, 6))
fig.suptitle("Class Distributions Across All Datasets", fontsize=16, fontweight='bold')

# --- AZH Distribution ---
azh_train_counts = {cls: 0 for cls in CFG["azh_classes"]}
azh_test_counts = {cls: 0 for cls in CFG["azh_classes"]}
for cls in CFG["azh_classes"]:
    azh_train_counts[cls] = len([f for f in os.listdir(os.path.join(CFG["azh_train_root"], cls))
                                  if os.path.splitext(f)[1].lower() in CFG["img_exts"]])
    azh_test_counts[cls] = len([f for f in os.listdir(os.path.join(CFG["azh_test_root"], cls))
                                 if os.path.splitext(f)[1].lower() in CFG["img_exts"]])

x = np.arange(len(CFG["azh_classes"]))
width = 0.35
bars1 = axes[0].bar(x - width/2, [azh_train_counts[c] for c in CFG["azh_classes"]], width, 
                     label='Train', color='#2196F3', edgecolor='white')
bars2 = axes[0].bar(x + width/2, [azh_test_counts[c] for c in CFG["azh_classes"]], width,
                     label='Test', color='#FF9800', edgecolor='white')
axes[0].set_xlabel("Class")
axes[0].set_ylabel("Count")
axes[0].set_title("AZH: Train vs Test Split", fontweight='bold')
axes[0].set_xticks(x)
axes[0].set_xticklabels(CFG["azh_classes"], rotation=30, ha='right', fontsize=9)
axes[0].legend()
# Add count labels on bars
for bar in bars1:
    axes[0].text(bar.get_x() + bar.get_width()/2., bar.get_height() + 1,
                 f'{int(bar.get_height())}', ha='center', va='bottom', fontsize=8)
for bar in bars2:
    axes[0].text(bar.get_x() + bar.get_width()/2., bar.get_height() + 1,
                 f'{int(bar.get_height())}', ha='center', va='bottom', fontsize=8)

# --- Medetec Distribution ---
med_counts = [len([f for f in os.listdir(os.path.join(CFG["medetec_root"], cls))
                    if os.path.splitext(f)[1].lower() in CFG["img_exts"]])
              for cls in CFG["medetec_classes"]]
colors_med = ['#4CAF50', '#F44336', '#9C27B0', '#00BCD4']
bars3 = axes[1].bar(CFG["medetec_classes"], med_counts, color=colors_med, edgecolor='white')
axes[1].set_xlabel("Class")
axes[1].set_ylabel("Count")
axes[1].set_title("Medetec: Class Distribution", fontweight='bold')
for bar in bars3:
    axes[1].text(bar.get_x() + bar.get_width()/2., bar.get_height() + 1,
                 f'{int(bar.get_height())}', ha='center', va='bottom', fontsize=9)

# --- Unified Distribution (what the transformer sees) ---
# Recompute from census
unified_counts = defaultdict(int)
for cls in CFG["azh_classes"]:
    unified_cls = CFG["wound_classes"][CFG["class_map"][cls]]
    unified_counts[unified_cls] += azh_train_counts[cls] + azh_test_counts[cls]
for cls in CFG["medetec_classes"]:
    unified_cls = CFG["wound_classes"][CFG["class_map"][cls]]
    unified_counts[unified_cls] += len([f for f in os.listdir(os.path.join(CFG["medetec_root"], cls))
                                         if os.path.splitext(f)[1].lower() in CFG["img_exts"]])

unified_names = CFG["wound_classes"]
unified_vals = [unified_counts[c] for c in unified_names]
colors_unified = ['#E91E63', '#FF5722', '#795548', '#009688', '#607D8B']
bars4 = axes[2].bar(unified_names, unified_vals, color=colors_unified, edgecolor='white')
axes[2].set_xlabel("Unified Class")
axes[2].set_ylabel("Count")
axes[2].set_title("UNIFIED: What the Transformer Sees (5-Class)", fontweight='bold')
for bar in bars4:
    axes[2].text(bar.get_x() + bar.get_width()/2., bar.get_height() + 2,
                 f'{int(bar.get_height())}', ha='center', va='bottom', fontsize=10, fontweight='bold')

# Add imbalance ratio annotation
max_cls = max(unified_vals)
min_cls = min(unified_vals)
axes[2].annotate(f"Imbalance ratio: {max_cls/min_cls:.1f}:1\n(venous:{max_cls} vs surgical:{min_cls})",
                 xy=(0.98, 0.95), xycoords='axes fraction', ha='right', va='top',
                 fontsize=9, bbox=dict(boxstyle='round,pad=0.3', facecolor='lightyellow'))

plt.tight_layout()
plt.savefig(os.path.join(CFG["figures_dir"], "panel2_class_distributions.png"), dpi=150, bbox_inches='tight')
plt.show()
print(f"  Saved: {os.path.join(CFG['figures_dir'], 'panel2_class_distributions.png')}")

# ══════════════════════════════════════════════════════════════════════════════
# 5. VISUALIZATION PANEL 3: IMAGE DIMENSION DISTRIBUTIONS
# ══════════════════════════════════════════════════════════════════════════════

print("\n\n📐 PANEL 3: IMAGE DIMENSION DISTRIBUTIONS")
print("-" * 80)

fig, axes = plt.subplots(1, 3, figsize=(20, 5))
fig.suptitle("Image Dimension Distributions per Dataset", fontsize=16, fontweight='bold')

# Collect dimensions (re-sample for visualization)
dataset_dims = {}
for dname, folder_fn in [
    ("FUSeg", lambda: [os.path.join(CFG["fuseg_train_img"], f) 
                        for f in os.listdir(CFG["fuseg_train_img"])
                        if os.path.splitext(f)[1].lower() in CFG["img_exts"]][:100]),
    ("AZH", lambda: [os.path.join(CFG["azh_train_root"], cls, f)
                      for cls in CFG["azh_classes"]
                      for f in os.listdir(os.path.join(CFG["azh_train_root"], cls))
                      if os.path.splitext(f)[1].lower() in CFG["img_exts"]][:200]),
    ("Medetec", lambda: [os.path.join(CFG["medetec_root"], cls, f)
                          for cls in CFG["medetec_classes"]
                          for f in os.listdir(os.path.join(CFG["medetec_root"], cls))
                          if os.path.splitext(f)[1].lower() in CFG["img_exts"]][:200]),
]:
    files = folder_fn()
    widths, heights = [], []
    for fpath in files:
        try:
            with Image.open(fpath) as img:
                w, h = img.size
                widths.append(w)
                heights.append(h)
        except:
            pass
    dataset_dims[dname] = (widths, heights)

for idx, (dname, (widths, heights)) in enumerate(dataset_dims.items()):
    if not widths:
        axes[idx].text(0.5, 0.5, "No data", ha='center', va='center', transform=axes[idx].transAxes)
        continue
    
    axes[idx].scatter(widths, heights, alpha=0.5, s=20, edgecolors='none')
    axes[idx].set_xlabel("Width (px)")
    axes[idx].set_ylabel("Height (px)")
    axes[idx].set_title(f"{dname} (n={len(widths)})", fontweight='bold')
    
    # Add reference lines for common sizes
    for ref_size in [224, 384, 512]:
        axes[idx].axhline(y=ref_size, color='red', linestyle='--', alpha=0.3, linewidth=0.8)
        axes[idx].axvline(x=ref_size, color='red', linestyle='--', alpha=0.3, linewidth=0.8)
    
    # Stats annotation
    w_arr, h_arr = np.array(widths), np.array(heights)
    axes[idx].annotate(f"W: [{w_arr.min()}-{w_arr.max()}] μ={w_arr.mean():.0f}\n"
                        f"H: [{h_arr.min()}-{h_arr.max()}] μ={h_arr.mean():.0f}",
                        xy=(0.02, 0.98), xycoords='axes fraction', ha='left', va='top',
                        fontsize=8, bbox=dict(boxstyle='round,pad=0.3', facecolor='white', alpha=0.8))

plt.tight_layout()
plt.savefig(os.path.join(CFG["figures_dir"], "panel3_dimension_distributions.png"), dpi=150, bbox_inches='tight')
plt.show()
print(f"  Saved: {os.path.join(CFG['figures_dir'], 'panel3_dimension_distributions.png')}")

# ══════════════════════════════════════════════════════════════════════════════
# 6. VISUALIZATION PANEL 4: FUSeg MASK ANALYSIS
# ══════════════════════════════════════════════════════════════════════════════

print("\n\n🔬 PANEL 4: FUSeg MASK ANALYSIS")
print("-" * 80)

# Compute wound area distribution from masks
train_mask_dir = CFG["fuseg_train_lbl"]
mask_files = sorted([f for f in os.listdir(train_mask_dir) 
                      if os.path.splitext(f)[1].lower() in CFG["img_exts"]])

wound_areas_pct = []
wound_pixels = []
empty_mask_files = []

for mf in tqdm(mask_files, desc="  Analyzing train masks"):
    mask = np.array(Image.open(os.path.join(train_mask_dir, mf)))
    if mask.ndim == 3:
        mask = mask[:, :, 0]
    wound_px = (mask > 0).sum()
    total_px = mask.shape[0] * mask.shape[1]
    area_pct = wound_px / total_px * 100
    wound_areas_pct.append(area_pct)
    wound_pixels.append(wound_px)
    if wound_px == 0:
        empty_mask_files.append(mf)

fig, axes = plt.subplots(1, 3, figsize=(20, 5))
fig.suptitle("FUSeg Mask Analysis — Wound Area Distribution", fontsize=16, fontweight='bold')

# Histogram of wound area percentages
wound_areas_nonzero = [a for a in wound_areas_pct if a > 0]
axes[0].hist(wound_areas_nonzero, bins=40, color='#E91E63', edgecolor='white', alpha=0.85)
axes[0].axvline(np.mean(wound_areas_nonzero), color='black', linestyle='--', linewidth=1.5,
                label=f'Mean: {np.mean(wound_areas_nonzero):.1f}%')
axes[0].axvline(np.median(wound_areas_nonzero), color='blue', linestyle='--', linewidth=1.5,
                label=f'Median: {np.median(wound_areas_nonzero):.1f}%')
axes[0].set_xlabel("Wound Area (% of image)")
axes[0].set_ylabel("Count")
axes[0].set_title("Wound Area Distribution (non-empty masks)", fontweight='bold')
axes[0].legend(fontsize=9)

# Box plot
axes[1].boxplot(wound_areas_nonzero, vert=True, widths=0.6,
                boxprops=dict(color='#E91E63'), medianprops=dict(color='black'),
                whiskerprops=dict(color='#E91E63'), capprops=dict(color='#E91E63'))
axes[1].set_ylabel("Wound Area (%)")
axes[1].set_title("Wound Area Box Plot", fontweight='bold')
stats_text = (f"n={len(wound_areas_nonzero)}\n"
              f"min={min(wound_areas_nonzero):.1f}%\n"
              f"Q1={np.percentile(wound_areas_nonzero, 25):.1f}%\n"
              f"median={np.median(wound_areas_nonzero):.1f}%\n"
              f"Q3={np.percentile(wound_areas_nonzero, 75):.1f}%\n"
              f"max={max(wound_areas_nonzero):.1f}%\n"
              f"empty masks={len(empty_mask_files)}")
axes[1].annotate(stats_text, xy=(0.98, 0.98), xycoords='axes fraction', ha='right', va='top',
                 fontsize=9, family='monospace',
                 bbox=dict(boxstyle='round,pad=0.5', facecolor='lightyellow', alpha=0.9))

# Show empty mask examples vs wound mask examples
# Pick 2 empty and 3 wound examples
sample_empty = empty_mask_files[:2] if empty_mask_files else []
wound_mask_files = [f for f in mask_files if f not in empty_mask_files]
sample_wound = random.sample(wound_mask_files, min(3, len(wound_mask_files)))

# Small grid inside axes[2]
axes[2].axis("off")
axes[2].set_title("Empty vs Wound Masks (samples)", fontweight='bold')

n_show = len(sample_empty) + len(sample_wound)
if n_show > 0:
    inner_grid = gridspec.GridSpecFromSubplotSpec(1, n_show, subplot_spec=axes[2].get_subplotspec(),
                                                  wspace=0.05)
    for i, mf in enumerate(sample_empty + sample_wound):
        ax_inner = fig.add_subplot(inner_grid[i])
        mask = np.array(Image.open(os.path.join(train_mask_dir, mf)))
        if mask.ndim == 3:
            mask = mask[:, :, 0]
        ax_inner.imshow(mask, cmap='hot', vmin=0, vmax=255)
        label = "EMPTY" if mf in empty_mask_files else f"{(mask>0).sum()/mask.size*100:.0f}%"
        color = 'red' if mf in empty_mask_files else 'green'
        ax_inner.set_title(label, fontsize=8, color=color, fontweight='bold')
        ax_inner.axis("off")

plt.tight_layout()
plt.savefig(os.path.join(CFG["figures_dir"], "panel4_mask_analysis.png"), dpi=150, bbox_inches='tight')
plt.show()
print(f"  Saved: {os.path.join(CFG['figures_dir'], 'panel4_mask_analysis.png')}")

# ══════════════════════════════════════════════════════════════════════════════
# 7. RESOLUTION DECISION SUMMARY
# ══════════════════════════════════════════════════════════════════════════════

print("\n\n" + "=" * 80)
print("  📋 CELL 2 SUMMARY — DATA INSIGHTS & RESOLUTION DECISIONS")
print("=" * 80)

print(f"""
  KEY FINDINGS:
  
  1. FUSeg: ALL images are 512×512 (uniform) ✅
     → Detection + Segmentation at 512×512 natively. No resizing waste.

  2. AZH: WILDLY variable (66px to 2624px, 178 unique combos)
     → Must resize to fixed resolution for classification training.
     → Many images are very small (~143-240px) — upscaling introduces artifacts.
     → Decision: Resize to 384×384 for Standard/XL, 224×224 for Mini.

  3. Medetec: Tiny (mostly 154×113)
     → These are the weakest images quality-wise.
     → Upscaling to 384×384 will be blurry but still informative for texture.
     → They supplement AZH for diabetic/pressure/venous classes.

  4. Class imbalance: venous(380) vs surgical(164) = 2.3:1
     → Weighted sampling + focal loss in classification training.

  5. FUSeg empty masks: train=10, val=14
     → These are our confirmed negatives for detection.
     → Combined with AZH BG(100) + no_wound(100) = 224 total negatives.

  6. FUSeg test: {"RESOLVED — see above" if all_test_images else "⚠️ 0 IMAGES FOUND — NEEDS INVESTIGATION"}

  All figures saved to: {CFG['figures_dir']}
  
  ✅ Ready for Cell 3: Build unified manifests (CSV files for all splits + tasks)
""")

✅ CFG loaded from Cell 1

🔍 INVESTIGATING FUSeg TEST FOLDER
--------------------------------------------------------------------------------
  Path: data/FUSeg/test
  Total entries: 1
  ├── Subdirectories: 1
  │   └── images/  (200 images, 200 total files)
  ├── Image files (direct): 0
  └── Non-image files: 0

  Recursive search under data/FUSeg/test:
  Total images found (recursive): 200
    images: 200 images

  ✅ RESOLVED: FUSeg test images found at: data/FUSeg/test/images
     Updating CFG['fuseg_test_img'] for subsequent cells.
     Dimensions (sampled 20): [((512, 512), 20)]


🖼️  PANEL 1: SAMPLE IMAGES FROM ALL DATASETS
--------------------------------------------------------------------------------


<Figure size 2400x960 with 10 Axes>

  Saved: artifacts/willie_v2/figures/panel1_fuseg_samples.png


<Figure size 1920x2160 with 24 Axes>

  Saved: artifacts/willie_v2/figures/panel1_azh_samples.png


...

<Figure size 2400x600 with 3 Axes>

  Saved: artifacts/willie_v2/figures/panel3_dimension_distributions.png


🔬 PANEL 4: FUSeg MASK ANALYSIS
--------------------------------------------------------------------------------


  Analyzing train masks:   0%|          | 0/610 [00:00<?, ?it/s]

<Figure size 2400x600 with 8 Axes>

  Saved: artifacts/willie_v2/figures/panel4_mask_analysis.png


  📋 CELL 2 SUMMARY — DATA INSIGHTS & RESOLUTION DECISIONS

  KEY FINDINGS:
  
  1. FUSeg: ALL images are 512×512 (uniform) ✅
     → Detection + Segmentation at 512×512 natively. No resizing waste.

  2. AZH: WILDLY variable (66px to 2624px, 178 unique combos)
     → Must resize to fixed resolution for classification training.
     → Many images are very small (~143-240px) — upscaling introduces artifacts.
     → Decision: Resize to 384×384 for Standard/XL, 224×224 for Mini.

  3. Medetec: Tiny (mostly 154×113)
     → These are the weakest images quality-wise.
     → Upscaling to 384×384 will be blurry but still informative for texture.
     → They supplement AZH for diabetic/pressure/venous classes.

  4. Class imbalance: venous(380) vs surgical(164) = 2.3:1
     → Weighted sampling + focal loss in classification training.

  5. FUSeg empty masks: train=10, val=14
     → These are our confirmed negatives for detection.
   

In [ ]:
"""
================================================================================
  CELL 3 — BUILD UNIFIED MANIFESTS (CSV FILES FOR ALL TASKS)
  Notebook: 01_WILLIE_DataForge.ipynb
================================================================================
  PURPOSE:
    • Build detection manifest: FUSeg images + masks → YOLO bbox + AZH negatives
    • Build classification manifest: AZH + Medetec → unified 5-class labels
    • Create stratified train/val/test splits for classification
    • Generate YOLO-format label files for detection training
    • Save all manifests as CSVs — single source of truth for all notebooks

  PRODUCES:
    manifests/det_train.csv    — detection training (FUSeg train + AZH neg)
    manifests/det_val.csv      — detection validation (FUSeg val)
    manifests/cls_train.csv    — classification training (AZH train + Medetec)
    manifests/cls_val.csv      — classification validation (AZH test subset)
    manifests/cls_test.csv     — classification test (AZH test subset)
    manifests/fuseg_test.csv   — FUSeg test images (no labels, for inference)

  DEPENDS ON: Cell 1 (CFG), Cell 2 (updated fuseg_test_img path)
================================================================================
"""

import os
import json
import random
import numpy as np
import pandas as pd
from pathlib import Path
from collections import Counter, defaultdict

from PIL import Image
from tqdm.notebook import tqdm
from sklearn.model_selection import train_test_split

# Verify CFG is loaded
assert 'CFG' in dir(), "❌ Run Cell 1 first — CFG not found."
print("✅ CFG loaded from Cell 1\n")

# ══════════════════════════════════════════════════════════════════════════════
# 1. DETECTION MANIFEST — FUSeg masks → YOLO bounding boxes
# ══════════════════════════════════════════════════════════════════════════════

print("🔧 BUILDING DETECTION MANIFESTS")
print("-" * 80)

def mask_to_bbox_yolo(mask_path, img_w=512, img_h=512):
    """
    Convert a binary mask to YOLO format bounding box.
    YOLO format: class_id x_center y_center width height (all normalized 0-1)
    
    Returns:
        list of [class_id, x_c, y_c, w, h] or empty list if mask is empty
    """
    mask = np.array(Image.open(mask_path))
    if mask.ndim == 3:
        mask = mask[:, :, 0]  # Convert 3-channel to single channel
    
    if mask.max() == 0:
        return [], 0  # Empty mask — negative sample
    
    # Find bounding box of non-zero region
    rows = np.any(mask > 0, axis=1)
    cols = np.any(mask > 0, axis=0)
    
    if not rows.any() or not cols.any():
        return [], 0
    
    y_min, y_max = np.where(rows)[0][[0, -1]]
    x_min, x_max = np.where(cols)[0][[0, -1]]
    
    # Convert to YOLO format (normalized)
    x_center = ((x_min + x_max) / 2.0) / img_w
    y_center = ((y_min + y_max) / 2.0) / img_h
    bbox_w = (x_max - x_min + 1) / img_w
    bbox_h = (y_max - y_min + 1) / img_h
    
    # Clamp to [0, 1]
    x_center = np.clip(x_center, 0, 1)
    y_center = np.clip(y_center, 0, 1)
    bbox_w = np.clip(bbox_w, 0, 1)
    bbox_h = np.clip(bbox_h, 0, 1)
    
    wound_pixels = (mask > 0).sum()
    
    # class 0 = wound
    return [[0, x_center, y_center, bbox_w, bbox_h]], wound_pixels


def build_fuseg_detection_manifest(split, img_dir, lbl_dir):
    """Build detection manifest for a FUSeg split."""
    records = []
    
    img_files = sorted([f for f in os.listdir(img_dir) 
                        if os.path.splitext(f)[1].lower() in CFG["img_exts"]])
    
    for fname in tqdm(img_files, desc=f"  FUSeg {split} → bbox"):
        img_path = os.path.join(img_dir, fname)
        
        # Find corresponding mask
        base = os.path.splitext(fname)[0]
        mask_path = None
        if lbl_dir:
            for ext in [".png", ".jpg", ".bmp", ".tif"]:
                candidate = os.path.join(lbl_dir, base + ext)
                if os.path.exists(candidate):
                    mask_path = candidate
                    break
        
        if mask_path and os.path.exists(mask_path):
            bboxes, wound_px = mask_to_bbox_yolo(mask_path)
            has_wound = len(bboxes) > 0
        else:
            bboxes = []
            wound_px = 0
            has_wound = False
        
        records.append({
            "image_path": img_path,
            "mask_path": mask_path if mask_path else "",
            "split": split,
            "has_wound": has_wound,
            "bbox_yolo": json.dumps(bboxes),  # Store as JSON string in CSV
            "wound_pixels": wound_px,
            "source": "FUSeg",
        })
    
    return records


# --- FUSeg train ---
det_train_records = build_fuseg_detection_manifest(
    "train", CFG["fuseg_train_img"], CFG["fuseg_train_lbl"]
)

# --- FUSeg val ---
det_val_records = build_fuseg_detection_manifest(
    "val", CFG["fuseg_val_img"], CFG["fuseg_val_lbl"]
)

# --- Add AZH negatives to detection training ---
print("\n  Adding AZH negative samples to detection training...")
azh_neg_count = 0
for cls in ["BG", "no wound"]:
    for split_name, split_root in [("train", CFG["azh_train_root"]), ("test", CFG["azh_test_root"])]:
        cls_dir = os.path.join(split_root, cls)
        if not os.path.isdir(cls_dir):
            continue
        for fname in os.listdir(cls_dir):
            if os.path.splitext(fname)[1].lower() in CFG["img_exts"]:
                det_train_records.append({
                    "image_path": os.path.join(cls_dir, fname),
                    "mask_path": "",
                    "split": "train",
                    "has_wound": False,
                    "bbox_yolo": json.dumps([]),
                    "wound_pixels": 0,
                    "source": f"AZH-{split_name}/{cls}",
                })
                azh_neg_count += 1

print(f"  Added {azh_neg_count} AZH negative images to detection training")

# --- Create DataFrames ---
det_train_df = pd.DataFrame(det_train_records)
det_val_df = pd.DataFrame(det_val_records)

# Shuffle training data
det_train_df = det_train_df.sample(frac=1, random_state=CFG["seed"]).reset_index(drop=True)

# Stats
train_pos = det_train_df["has_wound"].sum()
train_neg = len(det_train_df) - train_pos
val_pos = det_val_df["has_wound"].sum()
val_neg = len(det_val_df) - val_pos
neg_ratio_train = train_neg / len(det_train_df) * 100

print(f"\n  Detection Training:  {len(det_train_df)} total | {train_pos} wound | {train_neg} negative ({neg_ratio_train:.1f}%)")
print(f"  Detection Validation: {len(det_val_df)} total | {val_pos} wound | {val_neg} negative")

# ══════════════════════════════════════════════════════════════════════════════
# 2. GENERATE YOLO-FORMAT LABEL FILES FOR DETECTION TRAINING
# ══════════════════════════════════════════════════════════════════════════════

print("\n\n🏷️  GENERATING YOLO-FORMAT LABEL FILES")
print("-" * 80)

# Create YOLO dataset structure
yolo_base = os.path.join(CFG["output_root"], "det_yolo_dataset")
for sub in ["images/train", "images/val", "labels/train", "labels/val"]:
    os.makedirs(os.path.join(yolo_base, sub), exist_ok=True)

def write_yolo_labels(df, split_name):
    """Create symlinks for images and write YOLO label files."""
    img_out = os.path.join(yolo_base, "images", split_name)
    lbl_out = os.path.join(yolo_base, "labels", split_name)
    
    written = 0
    for _, row in tqdm(df.iterrows(), total=len(df), desc=f"  Writing {split_name} YOLO labels"):
        src_img = row["image_path"]
        # Use a unique filename: source_originalname
        src_base = os.path.basename(src_img)
        source_prefix = row["source"].replace("/", "_").replace(" ", "_").replace("-", "_")
        unique_name = f"{source_prefix}_{src_base}"
        
        # Ensure .png/.jpg extension for image
        img_ext = os.path.splitext(unique_name)[1]
        lbl_name = os.path.splitext(unique_name)[0] + ".txt"
        
        # Symlink image (avoid copying large files)
        dst_img = os.path.join(img_out, unique_name)
        if not os.path.exists(dst_img):
            try:
                os.symlink(src_img, dst_img)
            except OSError:
                # If symlink fails (e.g., cross-device), copy instead
                import shutil
                shutil.copy2(src_img, dst_img)
        
        # Write label file
        bboxes = json.loads(row["bbox_yolo"])
        dst_lbl = os.path.join(lbl_out, lbl_name)
        with open(dst_lbl, 'w') as f:
            for bbox in bboxes:
                # YOLO format: class x_center y_center width height
                f.write(f"{bbox[0]} {bbox[1]:.6f} {bbox[2]:.6f} {bbox[3]:.6f} {bbox[4]:.6f}\n")
            # Empty file for negatives (no bboxes) — YOLO expects this
        
        written += 1
    
    return written

n_train = write_yolo_labels(det_train_df, "train")
n_val = write_yolo_labels(det_val_df, "val")

print(f"\n  Written: {n_train} train + {n_val} val label files")
print(f"  YOLO dataset at: {yolo_base}")

# Write YOLO dataset YAML config
yolo_yaml_path = os.path.join(yolo_base, "dataset.yaml")
yolo_yaml_content = f"""# Willie Detection Dataset — Auto-generated
path: {yolo_base}
train: images/train
val: images/val

nc: 1
names: ['wound']
"""
with open(yolo_yaml_path, 'w') as f:
    f.write(yolo_yaml_content)
print(f"  YOLO config: {yolo_yaml_path}")

# Store in CFG for downstream notebooks
CFG["yolo_dataset_yaml"] = yolo_yaml_path
CFG["yolo_dataset_root"] = yolo_base

# ══════════════════════════════════════════════════════════════════════════════
# 3. CLASSIFICATION MANIFEST — AZH + Medetec → Unified 5-class
# ══════════════════════════════════════════════════════════════════════════════

print("\n\n🔧 BUILDING CLASSIFICATION MANIFESTS")
print("-" * 80)

cls_records = []

# --- AZH ---
for split_name, split_root, purpose in [
    ("azh_train", CFG["azh_train_root"], "train"),
    ("azh_test",  CFG["azh_test_root"],  "test"),
]:
    for cls in CFG["azh_classes"]:
        cls_dir = os.path.join(split_root, cls)
        if not os.path.isdir(cls_dir):
            continue
        unified_label = CFG["class_map"][cls]
        unified_name = CFG["wound_classes"][unified_label]
        
        for fname in os.listdir(cls_dir):
            if os.path.splitext(fname)[1].lower() in CFG["img_exts"]:
                cls_records.append({
                    "image_path": os.path.join(cls_dir, fname),
                    "original_class": cls,
                    "unified_label": unified_label,
                    "unified_class": unified_name,
                    "source": f"AZH-{split_name}",
                    "azh_split": purpose,  # preserve original AZH split info
                })

# --- Medetec (no predefined split — all goes to train pool) ---
for cls in CFG["medetec_classes"]:
    cls_dir = os.path.join(CFG["medetec_root"], cls)
    if not os.path.isdir(cls_dir):
        continue
    unified_label = CFG["class_map"][cls]
    unified_name = CFG["wound_classes"][unified_label]
    
    for fname in os.listdir(cls_dir):
        if os.path.splitext(fname)[1].lower() in CFG["img_exts"]:
            cls_records.append({
                "image_path": os.path.join(cls_dir, fname),
                "original_class": cls,
                "unified_label": unified_label,
                "unified_class": unified_name,
                "source": "Medetec",
                "azh_split": "train",  # Medetec goes to training
            })

cls_df = pd.DataFrame(cls_records)
print(f"  Total classification samples: {len(cls_df)}")
print(f"  Per class: {dict(cls_df['unified_class'].value_counts())}")
print(f"  Per source: {dict(cls_df['source'].value_counts())}")

# ══════════════════════════════════════════════════════════════════════════════
# 4. STRATIFIED TRAIN / VAL / TEST SPLIT FOR CLASSIFICATION
# ══════════════════════════════════════════════════════════════════════════════

print("\n\n📊 STRATIFIED SPLIT FOR CLASSIFICATION")
print("-" * 80)

# Strategy:
# - AZH has its own train/test split → respect it
# - Medetec has no split → goes to training pool
# - From combined training pool, hold out 15% as validation
# - AZH test set stays as our test set

# Training pool: AZH train + all Medetec
train_pool_df = cls_df[cls_df["azh_split"] == "train"].copy()
test_df = cls_df[cls_df["azh_split"] == "test"].copy()

print(f"  Training pool (before val split): {len(train_pool_df)}")
print(f"  Test set (AZH test):              {len(test_df)}")

# Stratified validation split from training pool
train_df, val_df = train_test_split(
    train_pool_df,
    test_size=0.15,
    random_state=CFG["seed"],
    stratify=train_pool_df["unified_label"]
)

train_df = train_df.reset_index(drop=True)
val_df = val_df.reset_index(drop=True)
test_df = test_df.reset_index(drop=True)

print(f"\n  Final splits:")
print(f"  ├── Train: {len(train_df)}")
print(f"  ├── Val:   {len(val_df)}")
print(f"  └── Test:  {len(test_df)}")

# Per-class breakdown per split
print(f"\n  Per-class breakdown:")
print(f"  {'Class':<15s} {'Train':>8s} {'Val':>8s} {'Test':>8s} {'Total':>8s}")
print(f"  {'─'*15} {'─'*8} {'─'*8} {'─'*8} {'─'*8}")
for cls_name in CFG["wound_classes"]:
    n_train = (train_df["unified_class"] == cls_name).sum()
    n_val = (val_df["unified_class"] == cls_name).sum()
    n_test = (test_df["unified_class"] == cls_name).sum()
    n_total = n_train + n_val + n_test
    print(f"  {cls_name:<15s} {n_train:>8d} {n_val:>8d} {n_test:>8d} {n_total:>8d}")

total_all = len(train_df) + len(val_df) + len(test_df)
print(f"  {'─'*15} {'─'*8} {'─'*8} {'─'*8} {'─'*8}")
print(f"  {'TOTAL':<15s} {len(train_df):>8d} {len(val_df):>8d} {len(test_df):>8d} {total_all:>8d}")

# ══════════════════════════════════════════════════════════════════════════════
# 5. COMPUTE CLASS WEIGHTS FOR WEIGHTED SAMPLING
# ══════════════════════════════════════════════════════════════════════════════

print("\n\n⚖️  CLASS WEIGHTS FOR BALANCED TRAINING")
print("-" * 80)

train_class_counts = train_df["unified_label"].value_counts().sort_index()
total_train = len(train_df)
n_classes = CFG["num_classes"]

# Inverse frequency weighting
class_weights = {}
for label_idx in range(n_classes):
    count = train_class_counts.get(label_idx, 1)
    weight = total_train / (n_classes * count)
    class_weights[label_idx] = round(weight, 4)
    cls_name = CFG["wound_classes"][label_idx]
    print(f"  Class {label_idx} ({cls_name:<12s}): count={count:>4d}, weight={weight:.4f}")

# Per-sample weights for WeightedRandomSampler
sample_weights = train_df["unified_label"].map(class_weights).values
print(f"\n  Sample weights range: [{sample_weights.min():.4f}, {sample_weights.max():.4f}]")

# Store in CFG
CFG["class_weights"] = class_weights
CFG["cls_train_size"] = len(train_df)
CFG["cls_val_size"] = len(val_df)
CFG["cls_test_size"] = len(test_df)

# ══════════════════════════════════════════════════════════════════════════════
# 6. FUSEG TEST MANIFEST (for inference later)
# ══════════════════════════════════════════════════════════════════════════════

print("\n\n📝 FUSeg TEST MANIFEST (for Stage 1+2 inference)")
print("-" * 80)

fuseg_test_records = []
fuseg_test_dir = CFG["fuseg_test_img"]
if os.path.isdir(fuseg_test_dir):
    for fname in sorted(os.listdir(fuseg_test_dir)):
        if os.path.splitext(fname)[1].lower() in CFG["img_exts"]:
            fuseg_test_records.append({
                "image_path": os.path.join(fuseg_test_dir, fname),
                "filename": fname,
            })

fuseg_test_df = pd.DataFrame(fuseg_test_records)
print(f"  FUSeg test images: {len(fuseg_test_df)}")

# ══════════════════════════════════════════════════════════════════════════════
# 7. SAVE ALL MANIFESTS
# ══════════════════════════════════════════════════════════════════════════════

print("\n\n💾 SAVING ALL MANIFESTS")
print("-" * 80)

manifest_files = {
    "det_train.csv":    det_train_df,
    "det_val.csv":      det_val_df,
    "cls_train.csv":    train_df,
    "cls_val.csv":      val_df,
    "cls_test.csv":     test_df,
    "fuseg_test.csv":   fuseg_test_df,
}

for fname, df in manifest_files.items():
    fpath = os.path.join(CFG["manifests_dir"], fname)
    df.to_csv(fpath, index=False)
    print(f"  ✅ {fname:<25s} → {len(df):>5d} rows  →  {fpath}")

# Update and re-save CFG
cfg_save_path = os.path.join(CFG["output_root"], "config.json")
cfg_serializable = {k: list(v) if isinstance(v, set) else v for k, v in CFG.items()}
with open(cfg_save_path, 'w') as f:
    json.dump(cfg_serializable, f, indent=2)
print(f"\n  Config updated: {cfg_save_path}")

# ══════════════════════════════════════════════════════════════════════════════
# 8. VISUALIZATION — SPLIT DISTRIBUTIONS
# ══════════════════════════════════════════════════════════════════════════════

print("\n\n📊 VISUALIZING SPLIT DISTRIBUTIONS")
print("-" * 80)

fig, axes = plt.subplots(1, 2, figsize=(16, 5))
fig.suptitle("Final Data Splits — Detection & Classification", fontsize=14, fontweight='bold')

# --- Detection split ---
det_data = {
    "Train (wound)": train_pos,
    "Train (neg.)": train_neg,
    "Val (wound)": val_pos,
    "Val (neg.)": val_neg,
}
colors_det = ['#2196F3', '#BBDEFB', '#FF9800', '#FFE0B2']
bars = axes[0].bar(det_data.keys(), det_data.values(), color=colors_det, edgecolor='white')
axes[0].set_title("Detection Splits", fontweight='bold')
axes[0].set_ylabel("Count")
for bar in bars:
    axes[0].text(bar.get_x() + bar.get_width()/2., bar.get_height() + 2,
                 f'{int(bar.get_height())}', ha='center', va='bottom', fontsize=10, fontweight='bold')
axes[0].tick_params(axis='x', rotation=20)

# --- Classification split per class ---
x = np.arange(len(CFG["wound_classes"]))
width = 0.25

train_counts = [int((train_df["unified_class"] == c).sum()) for c in CFG["wound_classes"]]
val_counts = [int((val_df["unified_class"] == c).sum()) for c in CFG["wound_classes"]]
test_counts = [int((test_df["unified_class"] == c).sum()) for c in CFG["wound_classes"]]

bars1 = axes[1].bar(x - width, train_counts, width, label=f'Train ({len(train_df)})', color='#4CAF50')
bars2 = axes[1].bar(x, val_counts, width, label=f'Val ({len(val_df)})', color='#FFC107')
bars3 = axes[1].bar(x + width, test_counts, width, label=f'Test ({len(test_df)})', color='#F44336')

axes[1].set_title("Classification Splits (Stratified)", fontweight='bold')
axes[1].set_ylabel("Count")
axes[1].set_xticks(x)
axes[1].set_xticklabels(CFG["wound_classes"], rotation=30, ha='right')
axes[1].legend(fontsize=9)

# Add count labels
for bars in [bars1, bars2, bars3]:
    for bar in bars:
        h = bar.get_height()
        if h > 0:
            axes[1].text(bar.get_x() + bar.get_width()/2., h + 1,
                         f'{int(h)}', ha='center', va='bottom', fontsize=7)

plt.tight_layout()
plt.savefig(os.path.join(CFG["figures_dir"], "panel5_split_distributions.png"), dpi=150, bbox_inches='tight')
plt.show()
print(f"  Saved: {os.path.join(CFG['figures_dir'], 'panel5_split_distributions.png')}")

# ══════════════════════════════════════════════════════════════════════════════
# 9. SUMMARY
# ══════════════════════════════════════════════════════════════════════════════

print("\n\n" + "=" * 80)
print("  📋 CELL 3 SUMMARY — ALL MANIFESTS BUILT")
print("=" * 80)

print(f"""
  DETECTION (Stage 1 — RT-DETR / YOLO):
    Train: {len(det_train_df)} images ({train_pos} wound + {train_neg} negative = {neg_ratio_train:.0f}% neg ratio)
    Val:   {len(det_val_df)} images ({val_pos} wound + {val_neg} negative)
    YOLO dataset: {yolo_base}

  CLASSIFICATION (Stage 3 — Willie Transformer):
    Train: {len(train_df)} images (stratified)
    Val:   {len(val_df)} images (stratified)
    Test:  {len(test_df)} images (AZH test set, held out)

  INFERENCE:
    FUSeg test: {len(fuseg_test_df)} images (no labels — for prediction)

  Class weights: {class_weights}

  All manifests in: {CFG['manifests_dir']}
  
  ✅ Ready for Cell 4: Augmentation pipeline + preview
""")

✅ CFG loaded from Cell 1

🔧 BUILDING DETECTION MANIFESTS
--------------------------------------------------------------------------------


  FUSeg train → bbox:   0%|          | 0/610 [00:00<?, ?it/s]

  FUSeg val → bbox:   0%|          | 0/400 [00:00<?, ?it/s]


  Adding AZH negative samples to detection training...
  Added 200 AZH negative images to detection training

  Detection Training:  810 total | 600 wound | 210 negative (25.9%)
  Detection Validation: 400 total | 386 wound | 14 negative


🏷️  GENERATING YOLO-FORMAT LABEL FILES
--------------------------------------------------------------------------------


  Writing train YOLO labels:   0%|          | 0/810 [00:00<?, ?it/s]

  Writing val YOLO labels:   0%|          | 0/400 [00:00<?, ?it/s]


  Written: 810 train + 400 val label files
  YOLO dataset at: artifacts/willie_v2/det_yolo_dataset
  YOLO config: artifacts/willie_v2/det_yolo_dataset/dataset.yaml


🔧 BUILDING CLASSIFICATION MANIFESTS
--------------------------------------------------------------------------------
  Total classification samples: 1314
  Per class: {'venous': np.int64(380), 'pressure': np.int64(304), 'diabetic': np.int64(266), 'no_wound': np.int64(200), 'surgical': np.int64(164)}
  Per source: {'AZH-azh_train': np.int64(696), 'Medetec': np.int64(384), 'AZH-azh_test': np.int64(234)}


📊 STRATIFIED SPLIT FOR CLASSIFICATION
--------------------------------------------------------------------------------
  Training pool (before val split): 1080
  Test set (AZH test):              234

  Final splits:
  ├── Train: 918
  ├── Val:   162
  └── Test:  234

  Per-class breakdown:
  Class              Train      Val     Test    Total
  ─────────────── ──────── ──────── ──────── ────────
  diabetic             187

<Figure size 1920x600 with 2 Axes>

  Saved: artifacts/willie_v2/figures/panel5_split_distributions.png


  📋 CELL 3 SUMMARY — ALL MANIFESTS BUILT

  DETECTION (Stage 1 — RT-DETR / YOLO):
    Train: 810 images (600 wound + 210 negative = 26% neg ratio)
    Val:   400 images (386 wound + 14 negative)
    YOLO dataset: artifacts/willie_v2/det_yolo_dataset

  CLASSIFICATION (Stage 3 — WILLIE Transformer):
    Train: 918 images (stratified)
    Val:   162 images (stratified)
    Test:  234 images (AZH test set, held out)

  INFERENCE:
    FUSeg test: 200 images (no labels — for prediction)

  Class weights: {0: np.float64(0.9818), 1: np.float64(0.8017), 2: np.float64(1.7654), 3: np.float64(0.68), 4: np.float64(1.4344)}

  All manifests in: artifacts/willie_v2/manifests
  
  ✅ Ready for Cell 4: Augmentation pipeline + preview



In [ ]:
"""
================================================================================
  CELL 4 — AUGMENTATION PIPELINE + PREVIEW (FINAL CELL OF NOTEBOOK 01)
  Notebook: 01_WILLIE_DataForge.ipynb
================================================================================
  PURPOSE:
    • Define augmentation pipelines for detection and classification
    • Preview augmented samples side-by-side with originals
    • Save augmentation config for reuse across all notebooks
    • Wrap up Notebook 01 — data foundation complete

  DEPENDS ON: Cell 1 (CFG), Cell 2, Cell 3 (manifests)
================================================================================
"""

import os
import json
import random
import numpy as np
import pandas as pd
from PIL import Image

import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec

# We'll use albumentations — the standard for medical imaging augmentation
try:
    import albumentations as A
    from albumentations.pytorch import ToTensorV2
    print("✅ albumentations loaded")
except ImportError:
    import subprocess, sys
    subprocess.check_call([sys.executable, "-m", "pip", "install", "albumentations", "-q"])
    import albumentations as A
    from albumentations.pytorch import ToTensorV2
    print("✅ albumentations installed and loaded")

assert 'CFG' in dir(), "❌ Run Cell 1 first."
print("✅ CFG loaded from Cell 1\n")

random.seed(CFG["seed"])
np.random.seed(CFG["seed"])

# ══════════════════════════════════════════════════════════════════════════════
# 1. DEFINE AUGMENTATION PIPELINES
# ══════════════════════════════════════════════════════════════════════════════

print("🔧 DEFINING AUGMENTATION PIPELINES")
print("-" * 80)

# ── DETECTION AUGMENTATION (for RT-DETR / YOLO training) ──
# Note: RT-DETR and YOLO have their own built-in augmentations.
# We define this for reference and for any custom preprocessing.
# The actual det. training will use the model's native augmentation.

det_train_transform = A.Compose([
    A.LongestMaxSize(max_size=512),
    A.PadIfNeeded(min_height=512, min_width=512, border_mode=0, value=0),
    A.HorizontalFlip(p=0.5),
    A.VerticalFlip(p=0.3),
    A.RandomRotate90(p=0.3),
    A.OneOf([
        A.ColorJitter(brightness=0.3, contrast=0.3, saturation=0.2, hue=0.05, p=1.0),
        A.CLAHE(clip_limit=3.0, tile_grid_size=(8, 8), p=1.0),
        A.RandomBrightnessContrast(brightness_limit=0.2, contrast_limit=0.2, p=1.0),
    ], p=0.7),
    A.OneOf([
        A.GaussNoise(var_limit=(5.0, 25.0), p=1.0),
        A.GaussianBlur(blur_limit=(3, 5), p=1.0),
        A.MedianBlur(blur_limit=3, p=1.0),
    ], p=0.3),
    A.ShiftScaleRotate(shift_limit=0.05, scale_limit=0.1, rotate_limit=15, 
                        border_mode=0, p=0.5),
], bbox_params=A.BboxParams(format='yolo', label_fields=['class_labels'], 
                             min_visibility=0.3))

det_val_transform = A.Compose([
    A.LongestMaxSize(max_size=512),
    A.PadIfNeeded(min_height=512, min_width=512, border_mode=0, value=0),
])

# ── CLASSIFICATION AUGMENTATION (for Willie Transformer) ──
# Aggressive augmentation because we only have ~918 training images

# Standard/XL resolution
cls_train_transform_384 = A.Compose([
    A.LongestMaxSize(max_size=384),
    A.PadIfNeeded(min_height=384, min_width=384, border_mode=0, value=0),
    A.HorizontalFlip(p=0.5),
    A.VerticalFlip(p=0.3),
    A.RandomRotate90(p=0.3),
    A.ShiftScaleRotate(shift_limit=0.08, scale_limit=0.15, rotate_limit=20,
                        border_mode=0, p=0.6),
    A.OneOf([
        A.ColorJitter(brightness=0.4, contrast=0.4, saturation=0.3, hue=0.05, p=1.0),
        A.CLAHE(clip_limit=4.0, tile_grid_size=(8, 8), p=1.0),
        A.RandomBrightnessContrast(brightness_limit=0.3, contrast_limit=0.3, p=1.0),
    ], p=0.8),
    A.OneOf([
        A.GaussNoise(var_limit=(5.0, 30.0), p=1.0),
        A.GaussianBlur(blur_limit=(3, 7), p=1.0),
        A.MotionBlur(blur_limit=5, p=1.0),
    ], p=0.3),
    A.OneOf([
        A.ElasticTransform(alpha=50, sigma=50 * 0.05, p=1.0),
        A.GridDistortion(num_steps=5, distort_limit=0.2, p=1.0),
        A.OpticalDistortion(distort_limit=0.1, shift_limit=0.1, p=1.0),
    ], p=0.3),
    A.CoarseDropout(max_holes=4, max_height=40, max_width=40,
                     min_holes=1, min_height=10, min_width=10,
                     fill_value=0, p=0.3),
    A.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225]),
    ToTensorV2(),
])

# Mini resolution
cls_train_transform_224 = A.Compose([
    A.LongestMaxSize(max_size=224),
    A.PadIfNeeded(min_height=224, min_width=224, border_mode=0, value=0),
    A.HorizontalFlip(p=0.5),
    A.VerticalFlip(p=0.3),
    A.RandomRotate90(p=0.3),
    A.ShiftScaleRotate(shift_limit=0.08, scale_limit=0.15, rotate_limit=20,
                        border_mode=0, p=0.6),
    A.OneOf([
        A.ColorJitter(brightness=0.4, contrast=0.4, saturation=0.3, hue=0.05, p=1.0),
        A.CLAHE(clip_limit=4.0, tile_grid_size=(8, 8), p=1.0),
        A.RandomBrightnessContrast(brightness_limit=0.3, contrast_limit=0.3, p=1.0),
    ], p=0.8),
    A.OneOf([
        A.GaussNoise(var_limit=(5.0, 30.0), p=1.0),
        A.GaussianBlur(blur_limit=(3, 5), p=1.0),
    ], p=0.3),
    A.CoarseDropout(max_holes=3, max_height=25, max_width=25,
                     min_holes=1, min_height=8, min_width=8,
                     fill_value=0, p=0.3),
    A.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225]),
    ToTensorV2(),
])

# Val/Test (both resolutions) — no augmentation, just resize + normalize
cls_val_transform_384 = A.Compose([
    A.LongestMaxSize(max_size=384),
    A.PadIfNeeded(min_height=384, min_width=384, border_mode=0, value=0),
    A.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225]),
    ToTensorV2(),
])

cls_val_transform_224 = A.Compose([
    A.LongestMaxSize(max_size=224),
    A.PadIfNeeded(min_height=224, min_width=224, border_mode=0, value=0),
    A.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225]),
    ToTensorV2(),
])

print("  ✅ Detection augmentation pipeline defined (512×512)")
print("  ✅ Classification augmentation pipeline defined (384×384 for Std/XL)")
print("  ✅ Classification augmentation pipeline defined (224×224 for Mini)")
print("  ✅ Validation transforms defined (resize + normalize only)")

# Summarize augmentation strategy
print(f"""
  Augmentation strategy:
  ┌─────────────────────────┬────────────────────────────────────────────┐
  │ Transform               │ Purpose for wound images                  │
  ├─────────────────────────┼────────────────────────────────────────────┤
  │ HorizontalFlip (0.5)    │ Wounds appear on left/right limbs         │
  │ VerticalFlip (0.3)      │ Foot wounds: top/bottom orientation       │
  │ RandomRotate90 (0.3)    │ Camera angle varies in clinical settings  │
  │ ShiftScaleRotate (0.6)  │ Wound not always centered in frame        │
  │ ColorJitter (0.8)       │ Clinical lighting varies dramatically     │
  │ CLAHE                   │ Enhances wound texture contrast           │
  │ GaussNoise (0.3)        │ Simulates low-quality phone cameras       │
  │ ElasticTransform (0.3)  │ Mild deformation — wounds aren't rigid    │
  │ CoarseDropout (0.3)     │ Simulates partial occlusion (bandages)    │
  └─────────────────────────┴────────────────────────────────────────────┘
""")

# ══════════════════════════════════════════════════════════════════════════════
# 2. PREVIEW — CLASSIFICATION AUGMENTATION (384×384)
# ══════════════════════════════════════════════════════════════════════════════

print("\n🖼️  PREVIEW: CLASSIFICATION AUGMENTATION (384×384)")
print("-" * 80)

# Create a preview transform WITHOUT normalization/tensor (for visualization)
cls_preview_transform = A.Compose([
    A.LongestMaxSize(max_size=384),
    A.PadIfNeeded(min_height=384, min_width=384, border_mode=0, value=0),
    A.HorizontalFlip(p=0.5),
    A.VerticalFlip(p=0.3),
    A.RandomRotate90(p=0.3),
    A.ShiftScaleRotate(shift_limit=0.08, scale_limit=0.15, rotate_limit=20,
                        border_mode=0, p=0.6),
    A.OneOf([
        A.ColorJitter(brightness=0.4, contrast=0.4, saturation=0.3, hue=0.05, p=1.0),
        A.CLAHE(clip_limit=4.0, tile_grid_size=(8, 8), p=1.0),
        A.RandomBrightnessContrast(brightness_limit=0.3, contrast_limit=0.3, p=1.0),
    ], p=0.8),
    A.OneOf([
        A.GaussNoise(var_limit=(5.0, 30.0), p=1.0),
        A.GaussianBlur(blur_limit=(3, 7), p=1.0),
        A.MotionBlur(blur_limit=5, p=1.0),
    ], p=0.3),
    A.OneOf([
        A.ElasticTransform(alpha=50, sigma=50 * 0.05, p=1.0),
        A.GridDistortion(num_steps=5, distort_limit=0.2, p=1.0),
        A.OpticalDistortion(distort_limit=0.1, shift_limit=0.1, p=1.0),
    ], p=0.3),
    A.CoarseDropout(max_holes=4, max_height=40, max_width=40,
                     min_holes=1, min_height=10, min_width=10,
                     fill_value=0, p=0.3),
])

# Pick one sample from each class
cls_train_df = pd.read_csv(os.path.join(CFG["manifests_dir"], "cls_train.csv"))

fig, axes = plt.subplots(len(CFG["wound_classes"]), 5, figsize=(20, len(CFG["wound_classes"]) * 3.5))
fig.suptitle("Classification Augmentation Preview (384×384)\nCol 1: Original | Cols 2-5: Augmented Versions", 
             fontsize=14, fontweight='bold', y=1.02)

for row, cls_name in enumerate(CFG["wound_classes"]):
    # Get one sample from this class
    class_samples = cls_train_df[cls_train_df["unified_class"] == cls_name]["image_path"].tolist()
    if not class_samples:
        continue
    sample_path = random.choice(class_samples)
    
    try:
        img = np.array(Image.open(sample_path).convert("RGB"))
    except Exception as e:
        print(f"  ⚠️ Could not load {sample_path}: {e}")
        continue
    
    # Original (just resized)
    original_resized = A.Compose([
        A.LongestMaxSize(max_size=384),
        A.PadIfNeeded(min_height=384, min_width=384, border_mode=0, value=0),
    ])(image=img)["image"]
    
    axes[row, 0].imshow(original_resized)
    axes[row, 0].set_title(f"{cls_name}\n(original)", fontsize=9, fontweight='bold')
    axes[row, 0].axis("off")
    
    # 4 augmented versions
    for col in range(1, 5):
        augmented = cls_preview_transform(image=img)["image"]
        axes[row, col].imshow(augmented)
        axes[row, col].set_title(f"aug v{col}", fontsize=8, color='gray')
        axes[row, col].axis("off")

plt.tight_layout()
plt.savefig(os.path.join(CFG["figures_dir"], "panel6_cls_augmentation_preview.png"), 
            dpi=150, bbox_inches='tight')
plt.show()
print(f"  Saved: {os.path.join(CFG['figures_dir'], 'panel6_cls_augmentation_preview.png')}")

# ══════════════════════════════════════════════════════════════════════════════
# 3. PREVIEW — DETECTION AUGMENTATION (512×512) WITH BBOXES
# ══════════════════════════════════════════════════════════════════════════════

print("\n\n🖼️  PREVIEW: DETECTION AUGMENTATION (512×512) WITH BBOXES")
print("-" * 80)

det_train_df = pd.read_csv(os.path.join(CFG["manifests_dir"], "det_train.csv"))
wound_samples = det_train_df[det_train_df["has_wound"] == True].sample(4, random_state=CFG["seed"])

# Preview transform (no normalize for visualization)
det_preview_transform = A.Compose([
    A.LongestMaxSize(max_size=512),
    A.PadIfNeeded(min_height=512, min_width=512, border_mode=0, value=0),
    A.HorizontalFlip(p=0.5),
    A.VerticalFlip(p=0.3),
    A.ShiftScaleRotate(shift_limit=0.05, scale_limit=0.1, rotate_limit=15,
                        border_mode=0, p=0.5),
    A.ColorJitter(brightness=0.3, contrast=0.3, saturation=0.2, hue=0.05, p=0.7),
], bbox_params=A.BboxParams(format='yolo', label_fields=['class_labels'],
                             min_visibility=0.3))

fig, axes = plt.subplots(4, 3, figsize=(15, 18))
fig.suptitle("Detection Augmentation Preview (512×512)\nCol 1: Original + GT bbox | Col 2-3: Augmented + Transformed bbox",
             fontsize=14, fontweight='bold', y=1.01)

for row_idx, (_, row) in enumerate(wound_samples.iterrows()):
    img = np.array(Image.open(row["image_path"]).convert("RGB"))
    bboxes = json.loads(row["bbox_yolo"])
    
    if not bboxes:
        continue
    
    yolo_bboxes = [[b[1], b[2], b[3], b[4]] for b in bboxes]  # x_c, y_c, w, h
    class_labels = [int(b[0]) for b in bboxes]
    
    def draw_yolo_bbox(ax, image, yolo_boxes, title=""):
        """Draw YOLO format bboxes on image."""
        ax.imshow(image)
        h, w = image.shape[:2]
        for box in yolo_boxes:
            xc, yc, bw, bh = box
            x1 = int((xc - bw/2) * w)
            y1 = int((yc - bh/2) * h)
            x2 = int((xc + bw/2) * w)
            y2 = int((yc + bh/2) * h)
            import matplotlib.patches as mpatches
            rect = mpatches.Rectangle((x1, y1), x2-x1, y2-y1, 
                                       linewidth=2, edgecolor='lime', facecolor='none')
            ax.add_patch(rect)
        ax.set_title(title, fontsize=9)
        ax.axis("off")
    
    # Original
    orig_resized = A.Compose([
        A.LongestMaxSize(max_size=512),
        A.PadIfNeeded(min_height=512, min_width=512, border_mode=0, value=0),
    ])(image=img)["image"]
    draw_yolo_bbox(axes[row_idx, 0], orig_resized, yolo_bboxes, "Original + GT bbox")
    
    # 2 augmented versions
    for col in range(1, 3):
        try:
            result = det_preview_transform(
                image=img, bboxes=yolo_bboxes, class_labels=class_labels
            )
            aug_img = result["image"]
            aug_bboxes = result["bboxes"]
            draw_yolo_bbox(axes[row_idx, col], aug_img, aug_bboxes, f"Augmented v{col}")
        except Exception as e:
            axes[row_idx, col].text(0.5, 0.5, f"Aug failed:\n{str(e)[:50]}", 
                                     ha='center', va='center', transform=axes[row_idx, col].transAxes)
            axes[row_idx, col].axis("off")

plt.tight_layout()
plt.savefig(os.path.join(CFG["figures_dir"], "panel7_det_augmentation_preview.png"),
            dpi=150, bbox_inches='tight')
plt.show()
print(f"  Saved: {os.path.join(CFG['figures_dir'], 'panel7_det_augmentation_preview.png')}")

# ══════════════════════════════════════════════════════════════════════════════
# 4. SAVE AUGMENTATION CONFIG (for reuse in other notebooks)
# ══════════════════════════════════════════════════════════════════════════════

print("\n\n💾 SAVING AUGMENTATION CONFIG")
print("-" * 80)

aug_config = {
    "det_imgsz": 512,
    "cls_imgsz_standard": 384,
    "cls_imgsz_mini": 224,
    "normalize_mean": [0.485, 0.456, 0.406],
    "normalize_std": [0.229, 0.224, 0.225],
    "augmentation_summary": {
        "detection": "HFlip(0.5), VFlip(0.3), Rot90(0.3), ColorJitter/CLAHE(0.7), "
                     "GaussNoise/Blur(0.3), ShiftScaleRot(0.5)",
        "classification": "HFlip(0.5), VFlip(0.3), Rot90(0.3), ShiftScaleRot(0.6), "
                          "ColorJitter/CLAHE(0.8), GaussNoise/Blur(0.3), "
                          "Elastic/Grid/Optical(0.3), CoarseDropout(0.3)",
    }
}

aug_config_path = os.path.join(CFG["output_root"], "augmentation_config.json")
with open(aug_config_path, 'w') as f:
    json.dump(aug_config, f, indent=2)
print(f"  ✅ Augmentation config saved: {aug_config_path}")

# ══════════════════════════════════════════════════════════════════════════════
# 5. NOTEBOOK 01 COMPLETION SUMMARY
# ══════════════════════════════════════════════════════════════════════════════

print("\n\n" + "=" * 80)
print("  🏁 NOTEBOOK 01: Willie_DataForge — COMPLETE")
print("=" * 80)

print(f"""
  WHAT WE BUILT:
  ──────────────
  ✅ Master config (CFG) — single source of truth for all paths & params
  ✅ Data census — every image counted, every dimension measured
  ✅ FUSeg test resolved — 200 images at test/images/
  ✅ 7 visualization panels — samples, distributions, dimensions, masks, augmentation
  ✅ Detection manifests — 810 train (26% negative) + 400 val
  ✅ YOLO dataset structure — symlinked images + label files + dataset.yaml
  ✅ Classification manifests — 918 train + 162 val + 234 test (stratified)
  ✅ Class weights computed — surgical 1.77x, no_wound 1.43x, venous 0.68x
  ✅ Augmentation pipelines — detection (512) + classification (224/384)
  ✅ FUSeg test manifest — 200 images for inference prediction

  KEY NUMBERS:
  ────────────
  FUSeg:   1210 images (610 train + 400 val + 200 test), 512×512, masks available
  AZH:     930 images (696 train + 234 test), variable dimensions (66-2624px)
  Medetec: 384 images, tiny (154×113), supplements classification
  Total:   2524 images across all datasets

  ARTIFACTS SAVED:
  ────────────────
  Config:      {CFG['output_root']}/config.json
  Manifests:   {CFG['manifests_dir']}/ (6 CSV files)
  YOLO data:   {CFG['yolo_dataset_root']}/
  Figures:     {CFG['figures_dir']}/ (7 panels)
  Aug config:  {aug_config_path}

  ═══════════════════════════════════════════════════════════════
  NEXT: Notebook 02 — 02_Willie_DetectionEngine.ipynb
        Train RT-DETR-L on wound detection + YOLOv8m comparison
  ═══════════════════════════════════════════════════════════════
""")

✅ albumentations loaded
✅ CFG loaded from Cell 1

🔧 DEFINING AUGMENTATION PIPELINES
--------------------------------------------------------------------------------
  ✅ Detection augmentation pipeline defined (512×512)
  ✅ Classification augmentation pipeline defined (384×384 for Std/XL)
  ✅ Classification augmentation pipeline defined (224×224 for Mini)
  ✅ Validation transforms defined (resize + normalize only)

  Augmentation strategy:
  ┌─────────────────────────┬────────────────────────────────────────────┐
  │ Transform               │ Purpose for wound images                  │
  ├─────────────────────────┼────────────────────────────────────────────┤
  │ HorizontalFlip (0.5)    │ Wounds appear on left/right limbs         │
  │ VerticalFlip (0.3)      │ Foot wounds: top/bottom orientation       │
  │ RandomRotate90 (0.3)    │ Camera angle varies in clinical settings  │
  │ ShiftScaleRotate (0.6)  │ Wound not always centered in frame        │
  │ ColorJitter (0.8)       │ Clinica

<Figure size 2400x2100 with 25 Axes>

  Saved: artifacts/willie_v2/figures/panel6_cls_augmentation_preview.png


🖼️  PREVIEW: DETECTION AUGMENTATION (512×512) WITH BBOXES
--------------------------------------------------------------------------------


<Figure size 1800x2160 with 12 Axes>

  Saved: artifacts/willie_v2/figures/panel7_det_augmentation_preview.png


💾 SAVING AUGMENTATION CONFIG
--------------------------------------------------------------------------------
  ✅ Augmentation config saved: artifacts/willie_v2/augmentation_config.json


  🏁 NOTEBOOK 01: WILLIE_DataForge — COMPLETE

  WHAT WE BUILT:
  ──────────────
  ✅ Master config (CFG) — single source of truth for all paths & params
  ✅ Data census — every image counted, every dimension measured
  ✅ FUSeg test resolved — 200 images at test/images/
  ✅ 7 visualization panels — samples, distributions, dimensions, masks, augmentation
  ✅ Detection manifests — 810 train (26% negative) + 400 val
  ✅ YOLO dataset structure — symlinked images + label files + dataset.yaml
  ✅ Classification manifests — 918 train + 162 val + 234 test (stratified)
  ✅ Class weights computed — surgical 1.77x, no_wound 1.43x, venous 0.68x
  ✅ Augmentation pipelines — detection (512) + classification (224/384)
  ✅ FUSeg test manifest — 